# 排序模型（离线评估精简版）

这个版本只保留当前最需要的离线评估链路：

1. `LGBMRanker`：group-wise 排序模型
2. `LGBMClassifier`：point-wise 点击分类模型
3. `DIN`：PyTorch 版深度兴趣模型
4. 加权融合

## 这个 notebook 现在做什么？

- 读取 `4.recall_all.ipynb` 和 `5.feature_engineering_all.ipynb` 产出的排序特征
- 在固定验证集上分别评估 `LGBMRanker`、`LGBMClassifier`、`DIN`
- 输出每个模型的 AUC / NDCG / MRR / HR
- 对三路模型结果做一个简单的加权融合

## 这次精简掉了什么？

- 五折交叉验证
- OOF 特征导出
- Stacking（二层逻辑回归）
- 在线提交 / 生成提交文件相关逻辑

这样 notebook 里的指标含义会更直接：看到的就是固定验证集上的离线评估结果。


In [1]:
import copy
import gc
import random
import time
import warnings
from collections import defaultdict
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm import tqdm

import lightgbm as lgb
from sklearn.metrics import ndcg_score, roc_auc_score
from sklearn.preprocessing import MinMaxScaler

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
    TORCH_AVAILABLE = True
    TORCH_IMPORT_ERROR = None
except ImportError as e:
    torch = nn = F = DataLoader = Dataset = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = e


def ensure_torch_available():
    if not TORCH_AVAILABLE:
        raise ImportError(
            '6.ranking.ipynb 的深度模型部分需要 PyTorch。请切换到 funrecjrc 内核。'
        ) from TORCH_IMPORT_ERROR


# ============================================================================
# 核心配置
# ============================================================================
SEED = 2020
DIN_EPOCHS = 2
DIN_BATCH_SIZE = 256
DIN_LR = 1e-3

np.random.seed(SEED)
random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

print('=' * 70)
print('🔧 排序模型 - 当前运行模式: offline only')
print('🔬 仅保留固定验证集上的离线评估')
print(f'🧠 PyTorch 可用: {"是" if TORCH_AVAILABLE else "否"}')
print('=' * 70)


🔧 排序模型 - 当前运行模式: offline only
🔬 仅保留固定验证集上的离线评估
🧠 PyTorch 可用: 是


## 读取排序特征


In [2]:
project_root = Path.cwd().resolve()
data_path = project_root / 'data' / 'raw' / 'news_recommendation'
save_path = project_root / 'data' / 'processed' / 'temp_results'

for path in [data_path, save_path]:
    path.mkdir(parents=True, exist_ok=True)

print('📊 当前 notebook: 仅做离线验证，不再生成提交文件')
print(f'📁 项目根目录: {project_root}')
print(f'📁 原始数据路径: {data_path}')
print(f'📁 中间结果路径: {save_path}')


📊 当前 notebook: 仅做离线验证，不再生成提交文件
📁 项目根目录: /Users/lixiang/Desktop/funrec-new-rec
📁 原始数据路径: /Users/lixiang/Desktop/funrec-new-rec/data/raw/news_recommendation
📁 中间结果路径: /Users/lixiang/Desktop/funrec-new-rec/data/processed/temp_results


## 数据泄漏修复说明

离线评估时，如果直接用完整点击日志按用户聚合历史，验证集里待预测的“最后一次点击”可能会混进历史序列，属于明显泄漏。

这个版本统一使用 `click_hist_all.csv` 来构造用户历史：

- 它来自 leave-last-out 之后的历史点击
- 不包含验证目标本身
- 同一用户更早时间点重复点击过同一篇文章，仍然视为真实历史，不算泄漏


In [3]:
print('📂 正在加载特征数据...')
trn_user_item_feats_df = pd.read_csv(save_path / 'trn_user_item_feats_df_all.csv')
trn_user_item_feats_df['click_article_id'] = trn_user_item_feats_df['click_article_id'].astype(int)
print(f'   训练集特征: {trn_user_item_feats_df.shape}')

val_user_item_feats_df = pd.read_csv(save_path / 'val_user_item_feats_df_all.csv')
val_user_item_feats_df['click_article_id'] = val_user_item_feats_df['click_article_id'].astype(int)
print(f'   验证集特征: {val_user_item_feats_df.shape}')

print('✅ 特征数据加载完成！')


📂 正在加载特征数据...
   训练集特征: (1426880, 31)
   验证集特征: (8000000, 31)
✅ 特征数据加载完成！


In [4]:
def norm_sim(sim_series, weight=0.0):
    sim_series = pd.Series(sim_series)
    min_sim = sim_series.min()
    max_sim = sim_series.max()
    if max_sim == min_sim:
        return pd.Series(np.ones(len(sim_series)) + weight, index=sim_series.index)
    return (sim_series - min_sim) / (max_sim - min_sim) + weight


def calculate_mrr_metrics(df, topk=5, score_col='pred_score'):
    df_sorted = df.sort_values(by=['user_id', score_col], ascending=[True, False])
    mrr_sum = 0.0
    hr_sum = 0.0
    user_count = 0
    pos_user_count = 0

    for _, group in df_sorted.groupby('user_id'):
        user_count += 1
        labels = group['label'].values
        if labels.sum() == 0:
            continue
        pos_user_count += 1
        pos_idx = np.where(labels == 1)[0]
        if len(pos_idx) > 0:
            rank = pos_idx[0] + 1
            if rank <= topk:
                mrr_sum += 1.0 / rank
                hr_sum += 1.0

    mrr = mrr_sum / user_count if user_count > 0 else 0.0
    hr = hr_sum / user_count if user_count > 0 else 0.0

    print(f'   MRR@{topk}: {mrr:.5f}')
    print(f'   HR@{topk}:  {hr:.5f}')
    print(f'   有正样本用户: {pos_user_count}/{user_count}')
    return mrr, hr


def cal_ndcg(labels, preds, user_id_list, k=5):
    group_score = defaultdict(list)
    group_truth = defaultdict(list)
    for idx, user_id in enumerate(user_id_list):
        group_score[user_id].append(preds[idx])
        group_truth[user_id].append(labels[idx])

    ndcg_list = []
    for user_id in group_score:
        y_true = np.array([group_truth[user_id]])
        y_score = np.array([group_score[user_id]])
        if np.sum(y_true) == 0:
            ndcg_list.append(0.0)
        else:
            ndcg_list.append(ndcg_score(y_true, y_score, k=k))
    return float(np.mean(ndcg_list)) if ndcg_list else 0.0


def safe_auc(labels, preds):
    labels = np.asarray(labels)
    preds = np.asarray(preds)
    if np.unique(labels).size < 2:
        return float('nan')
    return float(roc_auc_score(labels, preds))


def build_feature_cols(train_df, val_df=None):
    exclude_cols = {'user_id', 'click_article_id', 'label', 'hist_click_article_id', 'cate_list', 'article_id'}
    frames = [train_df]
    if val_df is not None:
        frames.append(val_df)

    common_cols = set(frames[0].columns)
    for frame in frames[1:]:
        common_cols &= set(frame.columns)

    preferred_order = [
        'sim0', 'time_diff0', 'word_diff0',
        'sim_max', 'sim_min', 'sim_sum', 'sim_mean',
        'score', 'rank', 'click_size', 'time_diff_mean', 'active_level',
        'click_environment', 'click_deviceGroup', 'click_os', 'click_country',
        'click_region', 'click_referrer_type', 'user_time_hob1', 'user_time_hob2',
        'words_hbo', 'category_id', 'created_at_ts', 'words_count',
        'is_cat_hab', 'article_hot_level', 'article_user_num', 'article_time_diff_mean',
    ]
    cols = [col for col in preferred_order if col in common_cols and col not in exclude_cols]
    extras = [col for col in train_df.columns if col in common_cols and col not in exclude_cols and col not in cols]
    return cols + extras


## LGB排序模型


In [5]:
trn_user_item_feats_df_rank_model = trn_user_item_feats_df.copy()
val_user_item_feats_df_rank_model = val_user_item_feats_df.copy()

lgb_cols = build_feature_cols(
    trn_user_item_feats_df_rank_model,
    val_user_item_feats_df_rank_model,
)
print(f'可用 LGB 特征数: {len(lgb_cols)}')
print(lgb_cols)

trn_user_item_feats_df_rank_model.sort_values(by=['user_id'], inplace=True)
g_train = trn_user_item_feats_df_rank_model.groupby('user_id', as_index=False).count()['label'].values

val_user_item_feats_df_rank_model.sort_values(by=['user_id'], inplace=True)
g_val = val_user_item_feats_df_rank_model.groupby('user_id', as_index=False).count()['label'].values


可用 LGB 特征数: 28
['sim0', 'time_diff0', 'word_diff0', 'sim_max', 'sim_min', 'sim_sum', 'sim_mean', 'score', 'rank', 'click_size', 'time_diff_mean', 'active_level', 'click_environment', 'click_deviceGroup', 'click_os', 'click_country', 'click_region', 'click_referrer_type', 'user_time_hob1', 'user_time_hob2', 'words_hbo', 'category_id', 'created_at_ts', 'words_count', 'is_cat_hab', 'article_hot_level', 'article_user_num', 'article_time_diff_mean']


In [6]:
lgb_ranker = lgb.LGBMRanker(
    boosting_type='gbdt',
    num_leaves=31,
    reg_alpha=0.0,
    reg_lambda=1.0,
    max_depth=-1,
    n_estimators=200,
    subsample=0.7,
    colsample_bytree=0.7,
    subsample_freq=1,
    learning_rate=0.05,
    min_child_weight=20,
    random_state=SEED,
    n_jobs=8,
    verbose=-1,
)

print('🚀 开始训练 LGB 排序模型...')
lgb_ranker.fit(
    trn_user_item_feats_df_rank_model[lgb_cols],
    trn_user_item_feats_df_rank_model['label'],
    group=g_train,
    eval_set=[(val_user_item_feats_df_rank_model[lgb_cols], val_user_item_feats_df_rank_model['label'])],
    eval_group=[g_val],
    eval_at=[1, 2, 3, 4, 5],
    eval_metric=['ndcg'],
)

val_preds = lgb_ranker.predict(
    val_user_item_feats_df_rank_model[lgb_cols],
    num_iteration=getattr(lgb_ranker, 'best_iteration_', None),
)
val_user_item_feats_df_rank_model['pred_score'] = val_preds
val_user_item_feats_df_rank_model[['user_id', 'click_article_id', 'pred_score']].to_csv(
    save_path / 'ranking_lgb_ranker_score_all.csv', index=False
)

ndcg_5 = cal_ndcg(
    val_user_item_feats_df_rank_model['label'].values,
    val_preds,
    val_user_item_feats_df_rank_model['user_id'].values,
    k=5,
)
print()
print('📈 【LGB 排序模型 - 验证集效果】')
print(f'   NDCG@5: {ndcg_5:.4f}')
print('📊 【LGB Ranker - 详细指标】')
calculate_mrr_metrics(val_user_item_feats_df_rank_model, topk=5)


🚀 开始训练 LGB 排序模型...

📈 【LGB 排序模型 - 验证集效果】
   NDCG@5: 0.2607
📊 【LGB Ranker - 详细指标】
   MRR@5: 0.22342
   HR@5:  0.37393
   有正样本用户: 27382/40000


(0.22342041666666046, 0.373925)

## LGB分类模型


In [7]:
lgb_classifier = lgb.LGBMClassifier(
    boosting_type='gbdt',
    num_leaves=31,
    reg_alpha=0.0,
    reg_lambda=1.0,
    max_depth=-1,
    n_estimators=300,
    subsample=0.7,
    colsample_bytree=0.7,
    subsample_freq=1,
    learning_rate=0.05,
    min_child_weight=20,
    random_state=SEED,
    n_jobs=8,
    verbose=-1,
)

print('🚀 开始训练 LGB 分类模型...')
lgb_classifier.fit(
    trn_user_item_feats_df_rank_model[lgb_cols],
    trn_user_item_feats_df_rank_model['label'],
    eval_set=[(val_user_item_feats_df_rank_model[lgb_cols], val_user_item_feats_df_rank_model['label'])],
    eval_metric=['auc'],
)

val_preds = lgb_classifier.predict_proba(
    val_user_item_feats_df_rank_model[lgb_cols],
    num_iteration=getattr(lgb_classifier, 'best_iteration_', None),
)[:, 1]
val_user_item_feats_df_rank_model['pred_score_cls'] = val_preds
val_user_item_feats_df_rank_model[['user_id', 'click_article_id', 'pred_score_cls']].to_csv(
    save_path / 'ranking_lgb_cls_score_all.csv', index=False
)

auc = safe_auc(val_user_item_feats_df_rank_model['label'], val_preds)
print()
print('📈 【LGB 分类模型 - 验证集效果】')
print(f'   AUC: {auc:.4f}')
print('📊 【LGB Classifier - 详细指标】')
calculate_mrr_metrics(val_user_item_feats_df_rank_model, topk=5, score_col='pred_score_cls')


🚀 开始训练 LGB 分类模型...

📈 【LGB 分类模型 - 验证集效果】
   AUC: 0.9625
📊 【LGB Classifier - 详细指标】
   MRR@5: 0.21920
   HR@5:  0.37170
   有正样本用户: 27382/40000


(0.21920041666665951, 0.3717)

## DIN模型

原 notebook 的 DIN 思路是：

- 先把 `user_id`、`click_article_id`、设备/地域等离散特征做 embedding
- 把用户历史点击序列 `hist_click_article_id` 和当前候选文章 `click_article_id` 做 attention
- 用 attention 聚合出来的“与当前候选相关的历史兴趣表示”去辅助点击预测

这里保留这条模型线，但把实现改成 **PyTorch 版 DIN**，这样就能在我们当前环境里直接跑通。


In [8]:
print('📂 加载用户历史点击行为数据...')
all_data = pd.read_csv(save_path / 'click_hist_all.csv')
print(f'   使用 leave-last-out 历史: {all_data.shape}')

hist_click = all_data[['user_id', 'click_article_id']].groupby('user_id').agg(list).reset_index()
his_behavior_df = pd.DataFrame({
    'user_id': hist_click['user_id'],
    'hist_click_article_id': hist_click['click_article_id'],
})

trn_user_item_feats_df_din_model = trn_user_item_feats_df.copy().merge(his_behavior_df, on='user_id', how='left')
val_user_item_feats_df_din_model = val_user_item_feats_df.copy().merge(his_behavior_df, on='user_id', how='left')

for frame in [trn_user_item_feats_df_din_model, val_user_item_feats_df_din_model]:
    frame['hist_click_article_id'] = frame['hist_click_article_id'].apply(
        lambda x: x if isinstance(x, list) else []
    )

print(f'✅ DIN 训练集: {trn_user_item_feats_df_din_model.shape}')
print(f'   DIN 验证集: {val_user_item_feats_df_din_model.shape}')


📂 加载用户历史点击行为数据...
   使用 leave-last-out 历史: (912623, 9)
✅ DIN 训练集: (1426880, 32)
   DIN 验证集: (8000000, 32)


In [9]:
din_sparse_fea = [
    'user_id', 'click_article_id', 'category_id',
    'click_environment', 'click_deviceGroup', 'click_os',
    'click_country', 'click_region', 'click_referrer_type',
]

din_dense_candidates = [
    'sim0', 'time_diff0', 'word_diff0', 'sim_max', 'sim_min', 'sim_sum', 'sim_mean',
    'score', 'rank', 'click_size', 'time_diff_mean', 'active_level',
    'user_time_hob1', 'user_time_hob2', 'words_hbo', 'words_count',
    'is_cat_hab', 'article_hot_level', 'article_user_num', 'article_time_diff_mean',
]

din_dense_fea = [col for col in din_dense_candidates if col in trn_user_item_feats_df_din_model.columns]
print(f'DIN sparse 特征数: {len(din_sparse_fea)}')
print(f'DIN dense 特征数: {len(din_dense_fea)}')

mm = MinMaxScaler()
for feat in din_dense_fea:
    trn_user_item_feats_df_din_model[feat] = mm.fit_transform(trn_user_item_feats_df_din_model[[feat]])
    val_user_item_feats_df_din_model[feat] = mm.transform(val_user_item_feats_df_din_model[[feat]])

print('✅ DIN dense 特征归一化完成')


DIN sparse 特征数: 9
DIN dense 特征数: 20
✅ DIN dense 特征归一化完成


In [10]:
ensure_torch_available()

emb_dim = 8
max_len = 50
all_frames = [trn_user_item_feats_df_din_model, val_user_item_feats_df_din_model]

sparse_maps = {}
for feat in din_sparse_fea:
    vals = pd.concat([f[feat].fillna(0).astype('int64') for f in all_frames], ignore_index=True)
    if feat == 'click_article_id':
        hist_vals = []
        for f in all_frames:
            for seq in f['hist_click_article_id']:
                hist_vals.extend([int(x) for x in seq])
        if len(hist_vals) > 0:
            vals = pd.concat([vals, pd.Series(hist_vals, dtype='int64')], ignore_index=True)
    _, uniques = pd.factorize(vals, sort=False)
    mapping = {int(val): int(idx) for idx, val in enumerate(uniques)}
    if feat == 'click_article_id':
        mapping = {k: v + 1 for k, v in mapping.items()}  # 0 留给 padding
    sparse_maps[feat] = mapping


def pad_sequences_np(sequences, maxlen, value=0):
    padded = np.full((len(sequences), maxlen), value, dtype=np.int64)
    for idx, seq in enumerate(sequences):
        if len(seq) == 0:
            continue
        trunc = seq[-maxlen:]
        padded[idx, -len(trunc):] = np.asarray(trunc, dtype=np.int64)
    return padded


def build_din_input(df):
    x = {}
    for feat in din_sparse_fea:
        x[feat] = df[feat].fillna(0).astype('int64').map(sparse_maps[feat]).fillna(0).astype('int64').values
    for feat in din_dense_fea:
        x[feat] = df[feat].astype('float32').values

    click_map = sparse_maps['click_article_id']
    mapped_hist = []
    for seq in df['hist_click_article_id'].tolist():
        mapped_hist.append([int(click_map.get(int(v), 0)) for v in seq])
    x['hist_click_article_id'] = pad_sequences_np(mapped_hist, maxlen=max_len, value=0)
    return x


sparse_vocab_sizes = {
    feat: (max(mapping.values()) + 1 if len(mapping) > 0 else 1)
    for feat, mapping in sparse_maps.items()
}
print('✅ DIN 特征映射构建完成')
for feat, vocab_size in sparse_vocab_sizes.items():
    print(f'   {feat}: vocab_size = {vocab_size}')


✅ DIN 特征映射构建完成
   user_id: vocab_size = 149760
   click_article_id: vocab_size = 22783
   category_id: vocab_size = 248
   click_environment: vocab_size = 3
   click_deviceGroup: vocab_size = 4
   click_os: vocab_size = 8
   click_country: vocab_size = 11
   click_region: vocab_size = 28
   click_referrer_type: vocab_size = 7


In [11]:
ensure_torch_available()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class DINAttentionPooling(nn.Module):
    def __init__(self, emb_dim, hidden_units=(64, 32)):
        super().__init__()
        layers = []
        input_dim = emb_dim * 4
        for units in hidden_units:
            layers.append(nn.Linear(input_dim, units))
            layers.append(nn.ReLU())
            input_dim = units
        layers.append(nn.Linear(input_dim, 1))
        self.att_mlp = nn.Sequential(*layers)

    def forward(self, query_emb, hist_emb, mask):
        query = query_emb.unsqueeze(1).expand_as(hist_emb)
        att_input = torch.cat([query, hist_emb, query - hist_emb, query * hist_emb], dim=-1)
        scores = self.att_mlp(att_input).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e9)
        all_pad = ~mask.any(dim=1)
        scores = scores.masked_fill(all_pad.unsqueeze(1), 0.0)
        weights = torch.softmax(scores, dim=1)
        weights = weights * mask.float()
        denom = weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
        weights = weights / denom
        pooled = torch.bmm(weights.unsqueeze(1), hist_emb).squeeze(1)
        pooled = pooled.masked_fill(all_pad.unsqueeze(1), 0.0)
        return pooled


class PyTorchDIN(nn.Module):
    def __init__(self, sparse_vocab_sizes, dense_dim, emb_dim=8, dropout=0.2):
        super().__init__()
        self.sparse_fea = list(sparse_vocab_sizes.keys())
        self.dense_fea = din_dense_fea
        self.embeddings = nn.ModuleDict()
        for feat, vocab_size in sparse_vocab_sizes.items():
            if feat == 'click_article_id':
                self.embeddings[feat] = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
            else:
                self.embeddings[feat] = nn.Embedding(vocab_size, emb_dim)

        self.attention = DINAttentionPooling(emb_dim, hidden_units=(64, 32))
        input_dim = len(self.sparse_fea) * emb_dim + dense_dim + emb_dim
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, batch):
        sparse_embs = [self.embeddings[feat](batch[feat]) for feat in self.sparse_fea]
        dense_tensor = torch.stack([batch[feat] for feat in din_dense_fea], dim=1)

        target_emb = self.embeddings['click_article_id'](batch['click_article_id'])
        hist_ids = batch['hist_click_article_id']
        hist_emb = self.embeddings['click_article_id'](hist_ids)
        hist_mask = hist_ids.ne(0)
        hist_repr = self.attention(target_emb, hist_emb, hist_mask)

        dnn_input = torch.cat(sparse_embs + [dense_tensor, hist_repr], dim=1)
        logits = self.mlp(dnn_input).squeeze(-1)
        return logits


class DictDataset(Dataset):
    def __init__(self, x_dict, y=None):
        self.x_dict = x_dict
        self.y = y
        self.length = len(next(iter(x_dict.values())))

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        sample = {key: value[idx] for key, value in self.x_dict.items()}
        if self.y is None:
            return sample
        return sample, self.y[idx]


def collate_with_labels(batch):
    features, labels = zip(*batch)
    collated = {}
    for key in features[0]:
        arr = np.stack([item[key] for item in features])
        dtype = torch.long if key in din_sparse_fea or key == 'hist_click_article_id' else torch.float32
        collated[key] = torch.tensor(arr, dtype=dtype)
    return collated, torch.tensor(np.asarray(labels), dtype=torch.float32)


def collate_features(batch):
    collated = {}
    for key in batch[0]:
        arr = np.stack([item[key] for item in batch])
        dtype = torch.long if key in din_sparse_fea or key == 'hist_click_article_id' else torch.float32
        collated[key] = torch.tensor(arr, dtype=dtype)
    return collated


def make_din_loader(x_dict, y=None, batch_size=256, shuffle=False):
    dataset = DictDataset(x_dict, y)
    if y is None:
        return DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_features)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_with_labels)


def move_batch_to_device(batch):
    return {key: value.to(DEVICE) for key, value in batch.items()}


def init_din_model():
    return PyTorchDIN(sparse_vocab_sizes, dense_dim=len(din_dense_fea), emb_dim=emb_dim).to(DEVICE)


print(f'📦 DIN 训练设备: {DEVICE}')


📦 DIN 训练设备: cpu


In [12]:
def predict_din(model, x_data, batch_size=256):
    loader = make_din_loader(x_data, y=None, batch_size=batch_size, shuffle=False)
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch_to_device(batch)
            logits = model(batch)
            probs = torch.sigmoid(logits)
            preds.append(probs.cpu().numpy())
    return np.concatenate(preds, axis=0)


def train_din_model(model, x_train, y_train, x_val=None, y_val=None, epochs=2, batch_size=256, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    train_loader = make_din_loader(x_train, y_train, batch_size=batch_size, shuffle=True)

    best_metric = float('-inf')
    best_state = None
    history = []

    for epoch in range(epochs):
        model.train()
        losses = []
        start = time.time()
        for x_batch, y_batch in train_loader:
            x_batch = move_batch_to_device(x_batch)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        train_loss = float(np.mean(losses)) if losses else float('nan')
        message = f'Epoch {epoch + 1}/{epochs} - train_loss: {train_loss:.4f}'

        if x_val is not None and y_val is not None:
            val_preds = predict_din(model, x_val, batch_size=batch_size)
            val_auc = safe_auc(y_val, val_preds)
            message += f' - val_auc: {val_auc:.4f}'
            metric = val_auc if not np.isnan(val_auc) else float('-inf')
            if metric > best_metric:
                best_metric = metric
                best_state = copy.deepcopy(model.state_dict())

        elapsed = time.time() - start
        message += f' ({elapsed:.1f}s)'
        history.append(message)
        print(message)

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


In [13]:
print('🚀 开始训练 PyTorch DIN 模型...')
x_trn_din = build_din_input(trn_user_item_feats_df_din_model)
y_trn_din = trn_user_item_feats_df_din_model['label'].values.astype('float32')

x_val_din = build_din_input(val_user_item_feats_df_din_model)
y_val_din = val_user_item_feats_df_din_model['label'].values.astype('float32')

main_model = init_din_model()
main_model, din_history = train_din_model(
    main_model,
    x_trn_din,
    y_trn_din,
    x_val=x_val_din,
    y_val=y_val_din,
    epochs=DIN_EPOCHS,
    batch_size=DIN_BATCH_SIZE,
    lr=DIN_LR,
)

val_preds = predict_din(main_model, x_val_din, batch_size=DIN_BATCH_SIZE)
val_res = val_user_item_feats_df_din_model[['user_id', 'click_article_id', 'label']].copy()
val_res['pred_score'] = val_preds
val_res['pred_rank'] = val_res.groupby('user_id')['pred_score'].rank(ascending=False, method='first')
val_res.to_csv(save_path / 'ranking_din_val_score_all.csv', index=False)

print()
print('📈 【PyTorch DIN - 验证集效果】')
print(f'   AUC: {safe_auc(val_res["label"], val_res["pred_score"]):.4f}')
print(f'   NDCG@5: {cal_ndcg(val_res["label"].values, val_res["pred_score"].values, val_res["user_id"].values, k=5):.4f}')
print('📊 【PyTorch DIN - 详细指标】')
calculate_mrr_metrics(val_res, topk=5)


🚀 开始训练 PyTorch DIN 模型...
Epoch 1/2 - train_loss: 0.2184 - val_auc: 0.9141 (293.1s)
Epoch 2/2 - train_loss: 0.2006 - val_auc: 0.9245 (288.3s)

📈 【PyTorch DIN - 验证集效果】
   AUC: 0.9245
   NDCG@5: 0.2097
📊 【PyTorch DIN - 详细指标】
   MRR@5: 0.17624
   HR@5:  0.31172
   有正样本用户: 27382/40000


(0.1762445833333245, 0.311725)

## 模型融合

### 加权融合


In [14]:
print('📂 加载各模型预测结果用于融合...')

lgb_ranker_df = pd.read_csv(save_path / 'ranking_lgb_ranker_score_all.csv').merge(
    val_user_item_feats_df[['user_id', 'click_article_id', 'label']],
    on=['user_id', 'click_article_id'], how='inner'
)
lgb_cls_df = pd.read_csv(save_path / 'ranking_lgb_cls_score_all.csv').rename(columns={'pred_score_cls': 'pred_score'}).merge(
    val_user_item_feats_df[['user_id', 'click_article_id', 'label']],
    on=['user_id', 'click_article_id'], how='inner'
)
din_ranker_df = pd.read_csv(save_path / 'ranking_din_val_score_all.csv').merge(
    val_user_item_feats_df[['user_id', 'click_article_id', 'label']],
    on=['user_id', 'click_article_id'], how='left'
)

rank_model = {
    'lgb_ranker': lgb_ranker_df,
    'lgb_cls': lgb_cls_df,
    'din_ranker': din_ranker_df,
}


def get_ensemble_predict_topk(rank_model, topk=5):
    print('🔗 开始加权融合模型预测结果...')
    base_df = rank_model['lgb_ranker'][['user_id', 'click_article_id', 'label', 'pred_score']].copy()
    base_df = base_df.rename(columns={'pred_score': 'pred_score_lgb_ranker'})

    lgb_cls_scores = rank_model['lgb_cls'][['user_id', 'click_article_id', 'pred_score']].copy()
    lgb_cls_scores = lgb_cls_scores.rename(columns={'pred_score': 'pred_score_lgb_cls'})
    base_df = base_df.merge(lgb_cls_scores, on=['user_id', 'click_article_id'], how='inner')

    din_scores = rank_model['din_ranker'][['user_id', 'click_article_id', 'pred_score']].copy()
    din_scores = din_scores.rename(columns={'pred_score': 'pred_score_din'})
    base_df = base_df.merge(din_scores, on=['user_id', 'click_article_id'], how='inner')

    if len(base_df) == 0:
        raise ValueError('三个模型没有共同的 user-item 对，无法融合。')

    s1 = norm_sim(base_df['pred_score_lgb_ranker'])
    s2 = norm_sim(base_df['pred_score_lgb_cls'])
    s3 = norm_sim(base_df['pred_score_din'])
    base_df['pred_score'] = 0.3 * s1 + 0.4 * s2 + 0.3 * s3

    base_df = base_df.dropna(subset=['label']).copy()
    print()
    print('📈 【融合模型 - 验证集效果】')
    print(f'   AUC: {safe_auc(base_df["label"], base_df["pred_score"]):.4f}')
    print(f'   NDCG@5: {cal_ndcg(base_df["label"].values, base_df["pred_score"].values, base_df["user_id"].values, k=5):.4f}')
    print('📊 【融合模型 - 详细指标】')
    calculate_mrr_metrics(base_df, topk=topk)


get_ensemble_predict_topk(rank_model)


📂 加载各模型预测结果用于融合...
🔗 开始加权融合模型预测结果...

📈 【融合模型 - 验证集效果】
   AUC: 0.9643
   NDCG@5: 0.2675
📊 【融合模型 - 详细指标】
   MRR@5: 0.22830
   HR@5:  0.38672
   有正样本用户: 27382/40000


## 总结

这个 notebook 现在只保留离线评估最核心的四块：

- `LGBMRanker`
- `LGBMClassifier`
- `DIN`（PyTorch 版）
- 加权融合

相对于之前的版本，这里刻意去掉了五折交叉验证、OOF 特征、Stacking 和在线提交逻辑。这样每一段输出的指标都更容易理解：它们就是固定验证集上的离线结果。
